# Cleaning Pipeline 01 — Geometry and Image-Quality Heuristics

This notebook finds images that may be difficult because of small dimensions, extreme aspect ratios, dark/bright exposure, low contrast, or possible blur.

These are **manual-review heuristics**, not automatic deletion rules.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "cleaning_pipeline":
    REPO_ROOT = REPO_ROOT.parents[1]
elif REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

sys.path.insert(0, str(REPO_ROOT / "src"))

DATA_ROOT = REPO_ROOT / "BDC2026"
OUTPUT_ROOT = REPO_ROOT / "eda_outputs" / "notebook_pipeline"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

LABEL2ID = {
    "0_Recyclable": 0,
    "1_Electronic": 1,
    "2_Organic": 2,
}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

print("REPO_ROOT:", REPO_ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from tqdm.auto import tqdm

manifest_path = OUTPUT_ROOT / "00_manifest" / "train_manifest.csv"
if not manifest_path.exists():
    raise FileNotFoundError(f"Run notebook 00 first: {manifest_path}")

manifest = pd.read_csv(manifest_path)
valid = manifest[manifest["is_valid"]].copy().reset_index(drop=True)
print("Valid images:", len(valid))

In [ ]:
MIN_SIDE = 80
ASPECT_LOW = 0.35
ASPECT_HIGH = 2.80
DARK_MEAN = 35
BRIGHT_MEAN = 220
LOW_CONTRAST_STD = 18
LOW_SHARPNESS = 20.0

def quality_metrics(path: str):
    with Image.open(path) as img:
        gray = np.asarray(ImageOps.exif_transpose(img).convert("L"), dtype=np.float32)
    gx = np.diff(gray, axis=1)
    gy = np.diff(gray, axis=0)
    return {
        "brightness_mean": float(gray.mean()),
        "contrast_std": float(gray.std()),
        "sharpness_score": float((np.mean(np.abs(gx)) + np.mean(np.abs(gy))) / 2.0),
    }

quality = pd.DataFrame([quality_metrics(p) for p in tqdm(valid["path"], desc="Quality metrics")])
valid = pd.concat([valid, quality], axis=1)

In [ ]:
valid["flag_small"] = valid["min_side"] < MIN_SIDE
valid["flag_extreme_aspect"] = (valid["aspect_ratio"] < ASPECT_LOW) | (valid["aspect_ratio"] > ASPECT_HIGH)
valid["flag_dark"] = valid["brightness_mean"] < DARK_MEAN
valid["flag_bright"] = valid["brightness_mean"] > BRIGHT_MEAN
valid["flag_low_contrast"] = valid["contrast_std"] < LOW_CONTRAST_STD
valid["flag_low_sharpness"] = valid["sharpness_score"] < LOW_SHARPNESS

flag_cols = [c for c in valid.columns if c.startswith("flag_")]
valid["num_quality_flags"] = valid[flag_cols].sum(axis=1)
display(valid[flag_cols].sum().sort_values(ascending=False).to_frame("candidate_count"))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
valid["brightness_mean"].hist(bins=50, ax=axes[0]); axes[0].axvline(DARK_MEAN, linestyle="--"); axes[0].axvline(BRIGHT_MEAN, linestyle="--"); axes[0].set_title("Brightness")
valid["contrast_std"].hist(bins=50, ax=axes[1]); axes[1].axvline(LOW_CONTRAST_STD, linestyle="--"); axes[1].set_title("Contrast")
valid["sharpness_score"].hist(bins=50, ax=axes[2]); axes[2].axvline(LOW_SHARPNESS, linestyle="--"); axes[2].set_title("Sharpness proxy")
plt.tight_layout(); plt.show()

In [ ]:
def show_grid(df, title, n=20, cols=5):
    sample = df.head(n)
    if len(sample) == 0:
        print("No candidates:", title)
        return
    rows = int(np.ceil(len(sample) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3.2, rows * 3.4))
    axes = np.asarray(axes).reshape(-1)
    for ax in axes: ax.axis("off")
    for ax, (_, row) in zip(axes, sample.iterrows()):
        with Image.open(row["path"]) as img:
            ax.imshow(ImageOps.exif_transpose(img).convert("RGB"))
        ax.set_title(f"{Path(row['path']).name}\n{row['class_name']}\nflags={int(row['num_quality_flags'])}", fontsize=8)
    fig.suptitle(title); plt.tight_layout(); plt.show()

candidates = valid[valid["num_quality_flags"] > 0].sort_values(["num_quality_flags", "sharpness_score"], ascending=[False, True])
show_grid(candidates, "Images with one or more quality flags")

## How to decide

Keep hard-but-valid examples when possible. Ask whether the object is still recognizable, whether the image is realistic for the test distribution, whether the label is plausible, and whether the issue is caused by preprocessing rather than the source image.

In [ ]:
stage_dir = OUTPUT_ROOT / "01_quality"
stage_dir.mkdir(parents=True, exist_ok=True)
valid.to_csv(stage_dir / "image_quality_metrics.csv", index=False)
candidates.to_csv(stage_dir / "image_quality_candidates.csv", index=False)
print("Saved outputs to", stage_dir)